# Práctica 1 (Parte 2)
## Uso de la distancia del coseno. Diseño de un sistema de recomendación de grupos musicales.

Disponemos de un dataset compuesto por 500 usuarios de una plataforma de música y sus preferencias musicales sobre 111 grupos y cantantes. Entendemos la preferencia musical como el numero de veces que el usuario ha escuchado una canción. Vamos a tratar de explotar esta información para responder a dos preguntas:

a) ¿Es posible obtener una categorización de los artistas en función de las preferencias de los usuarios? Es decir, podemos agrupar a los artistas por tipos de música que hacen siguiendo el criterio de que dos grupos hacen el mismo tipo de música si muchas veces se encuentran escuchados  juntos por los usuarios?

b) ¿Cuál es el usuario más parecido a otro usuario dado en función de sus preferencias? Esto nos servirá para recomendarle al primer usuario música que le gustó al segundo.

La distancia del coseno se utiliza para comparar vectores de características refiriéndose a su orientación en el espacio de caacterísticas. Esta basada en el angulo que forman ambos vectores.

In [16]:
# Usa esta celda para ir incorporando todos las librerías que necesitas
from sklearn.decomposition import NMF
from sklearn.preprocessing import Normalizer, MaxAbsScaler
from sklearn.pipeline import make_pipeline
import csv
import numpy as np
import sklearn.preprocessing as prepro
from scipy.spatial import distance
from sklearn.metrics.pairwise import cosine_distances


Se os proporcionan dos ficheros.
El primero se llama 'usuarios_musica.csv' y contiene por filas tres valores ej. [0, 76, 7] que interpretamos como que el usuario '0' ha escuchado el grupo/cantante '76' un número de '7' veces **(frecuencia)**. Hay en total 500 usuarios (con índices del 0 al 499) y 111 grupos y artistas (del 0 al 110). Este índice de artista coincide con el orden de la lista de grupos/artista que está almacenada en el fichero 'artistas.csv'

El fichero 'usuarios_musica.csv' tiene 2895 entradas que representan, como hemos dicho, cuántas veces ha escuchado un usuario un grupo. Hay del orden de 3, 4,5 o 6 elecciones por usuario, con lo que los datos son dispersos y, por tanto, no se tiene una información muy completa de los gustos de cada usuario.
Carga los datos de ambos ficheros en las estructuras que creas conveniente para la siguiente tarea que debes hacer (lee la caja de texto siguiente primero)

In [11]:
#Carga los datos de los ficheros. Llama 'escuchasmusicales' al contenido del fichero 'usuarios_musica.csv'
import pandas as pd
#codigo de carga aquí
escuchasmusicales = pd.read_csv("usuarios_musica.csv")
print(escuchasmusicales)

# cargamos en 'nombres_artistas' el contenido del fichero 'artistas.csv'. Elimina las cabeceras
filename="artists.csv"

with open(filename) as f:
    nombres_artistas = f.readlines()

#Quitamos los 'blancos' como el carácter '\n' al final de cada línea
nombres_artistas = [x.strip() for x in nombres_artistas] 

print(nombres_artistas)

      user_offset  artist_offset  playcount
0               1             79         58
1               1             84         80
2               1             86        317
3               1             89         64
4               1             96        159
...           ...            ...        ...
2889            0             75        371
2890            0             26         58
2891            0             52         58
2892            0             54         53
2893            0              1        128

[2894 rows x 3 columns]
['Massive Attack', 'Sublime', 'Beastie Boys', 'Neil Young', 'Dead Kennedys', 'Orbital', 'Miles Davis', 'Leonard Cohen', 'Van Morrison', 'NOFX', 'Rancid', 'Lamb', 'Korn', 'Dropkick Murphys', 'Bob Dylan', 'Eminem', 'Nirvana', 'Van Halen', 'Damien Rice', 'Elvis Costello', 'Everclear', 'Jimi Hendrix', 'PJ Harvey', 'Red Hot Chili Peppers', 'Ryan Adams', 'Soundgarden', 'The White Stripes', 'Madonna', 'Eric Clapton', 'Bob Marley', 'Dr. Dre', 'The Fla

La lista de artistas que debe salirte es:
['Massive Attack', 'Sublime', 'Beastie Boys', 'Neil Young', 'Dead Kennedys', 'Orbital', 'Miles Davis', 'Leonard Cohen', 'Van Morrison', 'NOFX', 'Rancid', 'Lamb', 'Korn', 'Dropkick Murphys', 'Bob Dylan', 'Eminem', 'Nirvana', 'Van Halen', 'Damien Rice', 'Elvis Costello', 'Everclear', 'Jimi Hendrix', 'PJ Harvey', 'Red Hot Chili Peppers', 'Ryan Adams', 'Soundgarden', 'The White Stripes', 'Madonna', 'Eric Clapton', 'Bob Marley', 'Dr. Dre', 'The Flaming Lips', 'Tom Waits', 'Moby', 'Cypress Hill', 'Garbage', 'Fear Factory', '50 Cent', 'Ani DiFranco', 'Matchbox Twenty', 'The Police', 'Eagles', 'Phish', 'Stone Temple Pilots', 'Black Sabbath', 'Britney Spears', 'Fatboy Slim', 'System of a Down', 'Simon & Garfunkel', 'Snoop Dogg', 'Aimee Mann', 'Less Than Jake', 'Rammstein', 'Reel Big Fish', 'The Prodigy', 'Pantera', 'Foo Fighters', 'The Beatles', 'Incubus', 'Audioslave', 'Bright Eyes', 'Machine Head', 'AC/DC', 'Dire Straits', 'MotÃ¶rhead', 'Ramones', 'Slipknot', 'Me First and the Gimme Gimmes', 'Bruce Springsteen', 'Queens of the Stone Age', 'The Chemical Brothers', 'Bon Jovi', 'Goo Goo Dolls', 'Alice in Chains', 'Howard Shore', 'Barenaked Ladies', 'Anti-Flag', 'Nick Cave and the Bad Seeds', 'Static-X', 'Misfits', '2Pac', 'Sparta', 'Interpol', 'The Crystal Method', 'The Beach Boys', 'Goldfrapp', 'Bob Marley & the Wailers', 'Kylie Minogue', 'The Blood Brothers', 'Mirah', 'Ludacris', 'Snow Patrol', 'The Mars Volta', 'Yeah Yeah Yeahs', 'Iced Earth', 'Fiona Apple', 'Rilo Kiley', 'Rufus Wainwright', 'Flogging Molly', 'Hot Hot Heat', 'Dredg', 'Switchfoot', 'Tegan and Sara', 'Rage Against the Machine', 'Keane', 'Jet', 'Franz Ferdinand', 'The Postal Service', 'The Dresden Dolls', 'The Killers', 'Death From Above 1979']

In [10]:
#Comprobamos que la dimensión de la matriz de 'escuchasmusicales' es la adecuada (2894x3)
print(escuchasmusicales.shape)


(2894, 3)


### Modelo de explotación de datos
Vamos a disponer la información en forma de una 'Bolsa de Palabras' porque vamos a emplear técnicas similares a las usadas en sistemas de comparación/obtención de textos similares.
La idea es la siguiente: Debes crear una estructura de datos tipo 'Bolsa de cantantes' donde, por columnas aparezcan los 111 grupos/cantantes (indices del 0 al 110) y por filas tengas los 500 usuarios (indices del 0 al 499)

Por ejemplo una fila de esta bolsa se pareceria a esto:
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0   90    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0 1201    0    0
    0    0    0   51    0    0    0  148    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0 2358    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0]

Que representa las elecciones de grupos de un usuario. Vemos que la mayoría de valores son 0 porque esos grupos/artistas no han sido elegidos por este usuario. El grupo de las 500 filas forma una **matriz dispersa** porque la mayoría de sus valores son 0.

Obtén esta matriz dispersa  de dimensión (500,111) a partir de los datos que has cargado en memoria del fichero 'usuarios_musica.csv'. Llámala 'matriz_usuarios'


**Cuidado!!** Considera que la lista puede no estar ordenada por usuarios (es decir que el usuario 0 puede no estar en las primeras filas del fichero (de hecho, está en las últimas))

In [15]:
#define la matriz de ceros
matriz_usuarios = np.zeros((500, 111))

#cargo la matriz y la llamo matriz_usuarios 
for index, row in escuchasmusicales.iterrows():
    usuario = row.iloc[0]   
    artista = row.iloc[1]    
    frecuencia = row.iloc[2] 
    matriz_usuarios[usuario, artista] = frecuencia

#Normaliza las caracteristicas de los datos usando MaxAbsScaler
#Esto nos proporciona un escalado a máximo 1.0 de cada característica
#evitando que características de frecuencias muy altas influencien mucho más que otras
#En este caso es equivalente a  MaxMinScale (que  desplaza los valores y los centra) aunque, en general, MaxAbsScaler no centra los datos
#con lo que se mantiene la dispersión de los datos
matriz_usuarios=MaxAbsScaler().fit_transform(matriz_usuarios)

#comprueba que sus dimensiones son correctas (dimensiones 500x111)
print(matriz_usuarios.shape)

(500, 111)


**Pregunta:** Has visto que hemos normalizado los datos por características.
¿Qué diferencia existe para el cálculo de la distancia del coseno entre esta normalización por características y aquella otra que normaliza por vectores (filas, es decir, usuarios ) de tal manera que el módulo del vector es 1?
(es decir, usar  *preprocessing.Normalize(matriz_usuarios)*? Ayuda: Piensa qué estas normalizando en este dataset cuando normalizas por columnaas usando *MaxAbsScaler()* y qué y cómo estas normalizando por filas al usar *Normalize()*. Piensa una situación sencilla donde cada fila es un vector bidimensional en un sistema cartesiano.
Responde en el cajetín  de abajo

In [ ]:
#Respuesta:

Ahora que ya tenemos cargada la matriz vamos a utilizar la **distancia del coseno** para calcular cuál es el usuario más parecido al usuario '0' . 

In [20]:
#Calcula la distancia del coseno de todos los usuarios de (1 a 499) al usuario 0 e imprime el índice del de menor distancia
#así como la distancia

# usuario 0
usuario0 = matriz_usuarios[0].reshape(1, -1)  # necesitamos forma 2D

# usuarios del 1 al 499
usuarios_restantes = matriz_usuarios[1:] 
distancias = cosine_distances(usuario0, usuarios_restantes) 
min_index = np.argmin(distancias)  
min_dist = distancias[0, min_index]
indice = min_index + 1

print("El más parecido al usuario 0 es el usuario", indice, "con valor distancia coseno:", min_dist)


El más parecido al usuario 0 es el usuario 30 con valor distancia coseno: 0.3229274440637857


Vamos a imprimir ahora la lista de grupos/cantantes ordenada por frecuencia para ambos usuarios y compararemos los primeros términos a ver cómo son de parecidos


In [28]:
#primero calcula la lista ordenada del usuario mas parecido
#Para ello, ordenamos de mayor a menor la frecuencia de los cantantes escuchados (no hay que hacer nada)
sorted_mylist = sorted(((v, i) for i, v in enumerate(matriz_usuarios[indice])), reverse=True)
#crea una lista ordenada de los indices de los cantantes más consultados en la lista de cantantes

#===> Pon tu código aquí
indices_ordenados_parecido = [i for v, i in sorted_mylist if v > 0]
print(indices_ordenados_parecido)
#segundo calcula la lista ordenada del usuario 0. Haz lo mismo que en el anterior
sorted_mylist_user0 = sorted(((v, i) for i, v in enumerate(matriz_usuarios[0])), reverse=True)
#crea una lista ordenada de los indices de los cantantes más consultados en la lista de cantantes

#===> Pon tu código aquí
indices_ordenados_user0 = [i for v, i in sorted_mylist_user0 if v > 0]

#Saco la lista de cantantes por orden de predilección del usuario 0 y la del más parecido. 
#Para ello, imprime las dos listas ordenadas de cantantes usando la lista ordenada de índices
#sacalo en un formato como este:
#--Usuario 0-->Barenaked Ladies --usuario parecido--> The Prodigy
#--Usuario 0-->The Prodigy --usuario parecido--> The White Stripes

#===> Pon tu código aquí
print("Comparación de los artistas más escuchados:\n")
for idx_user0, idx_parecido in zip(indices_ordenados_user0[:10], indices_ordenados_parecido[:10]):
    print(f"--Usuario 0 --> {nombres_artistas[idx_user0]} -- Usuario parecido --> {nombres_artistas[idx_parecido]}")

[54, 26, 1, 12, 36]
Comparación de los artistas más escuchados:

--Usuario 0 --> Barenaked Ladies -- Usuario parecido --> The Prodigy
--Usuario 0 --> The Prodigy -- Usuario parecido --> The White Stripes
--Usuario 0 --> The White Stripes -- Usuario parecido --> Sublime
--Usuario 0 --> Sublime -- Usuario parecido --> Korn
--Usuario 0 --> Rammstein -- Usuario parecido --> Fear Factory


Ten en cuenta que solamente serán válidos los primeros puestos (que coincidan con las entradas con datos de ambos usuarios (probablemente las primeras 4 ó 5). Como son vectores dispersos (la mayoria de las componentes son 0) el ordenamiento de artistas ya no tiene sentido para todos las frecuencias de valor 0.

**Pregunta:**
Indica qué recomendación sobre otros grupos le harías al usuario 0 

## Refinamiento del sistema de Recomendación. 
### Transformación de datos y Filtrado Colaborativo usando la factorización matricial NMF (Non-Negative Matrix Factorization)
Como has visto, la matriz que has creado en el ejercicio anterior es dispersa, es decir, sus entradas (por filas) tienen muchas componentes a 0. El **filtrado colaborativo** emplea una familia de técnicas para tratar de sustituir estos valores 0 por valores más informativos basados en el conjunto de datos. Una de las técnicas que se usan en los sistemas de recomendación y que tiene otras propiedades valiosas como vamos a ver, es la descomposición de la matriz de datos usando NMF.

La técnica de NMF no es una técnica de agrupamiento, sino una técnica de reducción de la dimensionalidad. Aquí solo la perfilaremos, pero podéis buscar en la literatura más información acerca de ella. El artículo original donde se discute NMF está accesible en internet: *D. D. Lee and H. S. Seung. Learning the parts of objects with nonnegative
matrix factorization. Nature, 401:788–791, 1999.*

NMF descompone la matriz de datos en el producto de dos matrices. Si llamamos $V$ a la matriz de datos (nuestra matriz_usuarios):
$$V = W H$$
donde cada matriz es interpretable. Así la matriz W nos da información para agrupar los datos en función de 'temas' (si los datos son textos' o, en nuestro caso por 'estilos musicales'). La matriz H se interpreta por columnas como una combinación lineal de esos estilos musicales que codifica a cada uno de los datos.

NMF toma una matriz semidefinida positiva V de dimensiones $n \times m$, y obtiene dos matrices W, H de tal manera que se cumple que:
$$ V_{n\times m} \approx W_{n \times k}~ H_{k \times m}$$

Fíjate que entre las dos partes de la expresión no hay un igual sino un aproximado. Esto tiene importancia ya que la reconstrucción de la matriz $V$ a partir de las matrices $W$ y $H$ da un resultado diferente al original, propiedad que explotaremos más adelante.

Un parámetro de dicha transformación es la dimensión intermedia $k$ que podemos interpretar como el número de 'temas ', o 'estilos musicales' en nuestro caso, que puede contener el dataset.


Python posee una implementación de la misma en "sklearn.decomposition.NMF"

Vamos a aplicar NMF a nuestro dataset. Para ello, nuestros datos deben estar primero normalizados  por características (que ya lo hemos hecho en el apartado anterior) y, tras la transformación NMF deben normalizarse cada uno de ellos.

Vamos a preparar  una tuberia (pipeline) con los dos últimos procesos, NMF y Normalizer ya que la matriz ya está normalizada por características.

In [36]:

# Crea un modelo NMF llamado nmf
#Aquí debes de configurar dos parámetros
#n_components que da idea de los 'estilos de grupos musicales 
#que existen (los 111 en cuantos estilos los clasificas)'
#max_iter EL número de iteraciones del algoritmo (entre 500 y 1000 puede ser razonable)

nmf = NMF(n_components=10, max_iter=800)

# Create a Normalizer: normalizer
normalizer = Normalizer()

# Create a pipeline: pipeline
#Ayuda: Si no estás failiarizado con las pipeline, puedes aplicar cada proceso por separado.
#APlica primero nmf y luefo normaliza el resultado
pipeline = make_pipeline(nmf, normalizer)
matriz_usuarios_nmf = pipeline.fit_transform(matriz_usuarios)
print(matriz_usuarios_nmf)

[[0.         0.28670172 0.12627291 ... 0.08828088 0.94451525 0.03050743]
 [0.0062249  0.         0.         ... 0.02277755 0.         0.        ]
 [0.47746428 0.         0.56842577 ... 0.32590058 0.27544239 0.        ]
 ...
 [0.00487804 0.         0.         ... 0.0016513  0.44312111 0.        ]
 [0.25307647 0.         0.         ... 0.10386392 0.06658307 0.        ]
 [0.         0.         0.02140301 ... 0.         0.1650616  0.        ]]


Antes de aplicar las transformaciones a nuestro dataset, vamos a trasponer la matriz para que los grupos musicales esten en filas y los usuarios en columnas. Esto lo hacemos para que la matriz W represente entonces las 'clases de musica diferentes' que representan los datos. Es decir, por columnas están los gustos de cada usuario caracterizados por las 111 bandas de música que están en filas.

Aplica a la tubería el método '*fit_transform()*'. No tendrás problemas ya que este método lo poseen las dos clases que forman la tubería.

Obtén las matrices W y H. Para ello consulta cómo se hace en la referencia OnLine de sklearn (mira el ejemplo que hay en esa página)

In [37]:
#Transpon la matriz  'matriz_usuarios'. Asígnasela a otra matriz llamada V
V = matriz_usuarios.T 

#Aplica ahora la tubería a la matriz V
W = pipeline.fit_transform(V)

H = pipeline.named_steps['nmf'].components_

#Comprueba que las dimensiones son correctas imprimiéndolas V = (111,500), W = (111,k) H = (k,500)
print(V.shape)
print(W.shape)
print(H.shape)

(111, 500)
(111, 10)
(10, 500)


Vamos a explotar la información de la matriz W. Esta matriz W es de dimensión  (111, n_components), donde n_components es el número de componentes que tú has configurado (y que debe ser la estimación de cuántos tipos de estilos musicales hay en el dataset). Esta matriz tiene caracterizados los grupos de música por estilos.

Fíjate que para cada grupo musical (filas) hemos 'pesado' a través de las columnas su pertenencia en cada estilo musical.

Vamos a sacar los grupos que el sistema nos dice que son parecidos a un grupo en concreto. Para ello elige un grupo de la lista, obtén su índice y recorre ahora todas las filas de la matriz W calculando para cada fila su distancia del coseno a la del artista elegido, he imprime por pantalla todos los grupos/artistas cuya distancia del coseno sea menor que una dada.

(Ejemplo, a mí me ha dado bien con el grupo 'Motorhead' (que es Heavy Metal) y una distancia menor que 0.1. Pero esto depende tambien del número de n_components que hayas elegido). Juega con los valores y juzga la calidad de tu sistema.

In [44]:
#Artistas parecidos a un artista concreto (Explotación de W)

indice_Artista =nombres_artistas.index('Eminem')

vector_artista = W[indice_Artista].reshape(1, -1)

# Distancias de coseno con todos los demás artistas
distancias = cosine_distances(vector_artista, W).flatten()

# Umbral de similitud
umbral = 0.1

print(f"Artistas similares a {nombres_artistas[indice_Artista]} (distancia < {umbral}):")
for i, d in enumerate(distancias):
    if i != indice_Artista and d < umbral:
        print(f"{nombres_artistas[i]} - distancia: {d:.3f}")

Artistas similares a Eminem (distancia < 0.1):
Korn - distancia: 0.080
Dr. Dre - distancia: 0.011
Cypress Hill - distancia: 0.000
System of a Down - distancia: 0.013
The Prodigy - distancia: 0.034
Static-X - distancia: 0.061
Ludacris - distancia: 0.019
Jet - distancia: 0.002


Ahora vamos a reconstruir la matriz V (que es la traspuesta de matriz_usuarios) a partir de W y H, usando el método inverse_transform(W) de la clase nmf.

In [46]:
new_V = nmf.inverse_transform(W)
print("new_V shape:", new_V.shape) 

new_V shape: (111, 500)


Transponemos la matriz new_V para obtener una matriz aproximada a la matriz 'matriz_usuarios'. La llamamos 'nueva_matriz_usuarios'

In [48]:
#trasposición de new_V

nueva_matriz_usuarios = new_V.T

#verificamos sus dimensiones  (500,111)
print(nueva_matriz_usuarios.shape)

print("Vector de características del usuario 0 (aproximado):")
print(nueva_matriz_usuarios[0])

(500, 111)
Vector de características del usuario 0 (aproximado):
[4.56010877e-03 4.02391108e-03 1.98170386e-02 1.88852142e-02
 4.34358843e-03 2.77073313e-02 1.06060518e-03 3.00151305e-03
 8.11831772e-04 0.00000000e+00 5.36209058e-04 4.88987133e-03
 3.94338357e-02 1.39629396e-03 7.82068455e-04 3.88223575e-02
 2.29387854e-03 4.71122278e-03 2.59048560e-03 2.92457751e-05
 8.89796785e-03 1.72861275e-03 2.62229773e-03 4.09723100e-03
 4.04565538e-04 5.21526990e-04 2.24942191e-02 4.83989475e-03
 9.84523252e-03 1.05727412e-03 4.02213702e-02 1.19750446e-02
 3.06099180e-03 6.80563747e-03 3.88301057e-02 1.55636604e-03
 1.23164323e-02 4.28943811e-03 1.37975893e-04 6.88346106e-03
 1.60702827e-03 1.47060683e-03 1.39652955e-03 1.12832593e-03
 1.39877210e-02 1.71830507e-03 6.28970854e-03 3.99103113e-02
 4.66909397e-03 6.17461478e-03 3.86610445e-03 2.15666811e-04
 1.41983800e-02 2.52626655e-03 4.03486901e-02 1.23164323e-02
 1.54887192e-03 2.04180071e-03 2.63733064e-03 1.68470153e-03
 8.01666223e-04 1.23

Puedes imprimir en la caja de código anterior el contenido del primer usuario de esta nueva matriz. **FIJATE** que ahora ya no es un vector disperso, sino que sus componentes ya no son cero. Hemos conseguido rellenar con información útil características que no tenía el dataset original,  a partir de todo el conjunto de datos. A esto se le llama aplicar una técnica de **filtrado colaborativo**.
Es lógico pensar que ahora los vectores de característias que representan los gustos de un usuario son más informativos que los que hemos hecho en el apartado anterior.


Ahora vamos a proceder a obtener el usuario más parecido al usuario '0' repitiendo el código del apartado anterior (es decir, calculando las distancias del coseno entre el vector de características del usuario '0' y el resto de los 499 usuarios. 


In [50]:
#A partir de aqui es como antes. Uso la distancia del coseno sobre 
#la nueva matriz.
#Voy a calcular el usuario más parecido en gustos al usuario 0 
#usando la distancia del coseno
usuario0 = nueva_matriz_usuarios[0].reshape(1, -1)
usuarios_restantes = nueva_matriz_usuarios[1:]
distancias = cosine_distances(usuario0, usuarios_restantes).flatten()
min_index = np.argmin(distancias)
min_dist = distancias[min_index]
indice = min_index + 1

print("el mas parecido al usuario 0 es el usuario", indice, "con valor distancia coseno: ", min_dist)

el mas parecido al usuario 0 es el usuario 475 con valor distancia coseno:  0.015547298666910625


Imprime por pantalla igual que antes la lista comparativa de grupos entre los dos usuarios. Ahora sí puedes comparar todos los grupos (no los n primeros), ya que no son vectores dispersos.

In [52]:
#saco la lista de cantantes por orden de predilección del usuario 0 y la del más parecido

usuario_parecido = indice

sorted_indices_user0 = np.argsort(-nueva_matriz_usuarios[0]) 

sorted_indices_parecido = np.argsort(-nueva_matriz_usuarios[usuario_parecido])

print("Comparación de artistas según gustos aproximados:\n")
for idx_user0, idx_parecido in zip(sorted_indices_user0, sorted_indices_parecido):
    print(f"--Usuario 0 --> {nombres_artistas[idx_user0]} -- Usuario parecido --> {nombres_artistas[idx_parecido]}")





Comparación de artistas según gustos aproximados:

--Usuario 0 --> Static-X -- Usuario parecido --> Dr. Dre
--Usuario 0 --> The Prodigy -- Usuario parecido --> Static-X
--Usuario 0 --> Dr. Dre -- Usuario parecido --> System of a Down
--Usuario 0 --> System of a Down -- Usuario parecido --> The Prodigy
--Usuario 0 --> Korn -- Usuario parecido --> Cypress Hill
--Usuario 0 --> Cypress Hill -- Usuario parecido --> Eminem
--Usuario 0 --> Eminem -- Usuario parecido --> Korn
--Usuario 0 --> Jet -- Usuario parecido --> Jet
--Usuario 0 --> Ludacris -- Usuario parecido --> Ludacris
--Usuario 0 --> Orbital -- Usuario parecido --> Goo Goo Dolls
--Usuario 0 --> Goo Goo Dolls -- Usuario parecido --> Orbital
--Usuario 0 --> The White Stripes -- Usuario parecido --> The White Stripes
--Usuario 0 --> Beastie Boys -- Usuario parecido --> Neil Young
--Usuario 0 --> Neil Young -- Usuario parecido --> Slipknot
--Usuario 0 --> Slipknot -- Usuario parecido --> Beastie Boys
--Usuario 0 --> Rage Against the Ma

Si todo ha ido bien, los grupos de ambas listas deben situarse los mismos en posiciones cercanas de las listas. Aunque conforme nos acercamos al final de las listas, hay más dispersión porque las preferencias ya no son tan claras.

**Acabas de diseñar un sistema de recomendación de dos vías: (por estilos musicales y por usuarios con gustos parecidos) basado en los gustos de 500 usuarios.**

Como has visto un parámetro importante del modelo es la definición del valor de la variable 'n_components' que indica en número de estilos musicales en que vamos a clasificar a los grupos de música. Haz un pequeño estudio, variando este valor y procediendo 
igual que has hecho hasta aquí para ver si el sistema de recomendación mejora. 
**Describe las pruebas que has realizado** (los valores que has elegido de esta variable) y los resultados de las mismas aquí:


